# PHYG 2026 — TME 4
### Phylogénie et génomique comparative
## Trees from many genes: single domains, bootstrap and concatenation

| | |
|---|---|
| **Student 1** | _surname, first name_ |
| **Student 2** | _surname, first name_ |
| **Date** | _dd/mm/yyyy_ |

Léon Faure — <leon.faure@sorbonne-universite.fr>
Course materials: <https://github.com/leonfaure/PHYG2026>

> ### Overview
>
> So far each tree came from a single gene. Here you build trees for **40
> eukaryotes** from ribosomal protein domains, first from one domain family, then
> from many families combined into one alignment. You meet a second tree-building
> method, **maximum likelihood** (PHYLIP's `proml`), next to Neighbor-Joining; you
> measure how much the data support each branch with the **bootstrap**; and you
> compare trees of different genes of the same viruses.
>
> **Learning items**:
> - explain why trees of thousands of species are hard to build, and how data from many genes are combined;
> - extract the sequences of a given set of species from protein-family files, align them, and build NJ and maximum-likelihood trees;
> - read an unrooted tree: decide whether the species of a clade are grouped together;
> - compute bootstrap support values and say what they do and do not mean;
> - combine several genes into one tree, and (bonus) check first that they *can* be combined.

> ### About these sessions
>
> The TMEs account for only **10% of the final grade**. They are there to help you
> understand the course material and to get hands-on experience with real
> problems — not to produce a perfect report. Please do the thinking yourself: it
> is **discouraged to let an AI do the work for you**. Using one to help you
> *understand* a concept, a tool or an error message is fine; handing it the
> questions and pasting back the answers defeats the purpose.

---

### How to hand this in

**Deadline: 3 December.** Work in groups of **at most two students**; you may
also work alone.

1. Fill in your names in the table above.
2. Answer **every** question (the bonus, Q4.3–Q4.4, is optional) — in the
   `Your answer` markdown cells for written answers, in the `TODO` code cells
   for code. Where a question only asks you to display something, the figure
   itself is the answer.
3. **Restart the kernel and run all cells** (`Kernel → Restart Kernel and Run All Cells`)
   and check it runs top to bottom without errors. The whole notebook takes a
   few minutes: maximum likelihood is slow.
4. Rename the file `SURNAME1_SURNAME2_TME4.ipynb` and e-mail it to
   <leon.faure@sorbonne-universite.fr> with the subject `[PHYG] TME4`.

Send **the notebook only** — no PDF, no archive. Figures produced by your code
are stored inside the `.ipynb` as cell outputs, so they travel with it.

***
## Toolbox

**Run the cell below first.** It is provided — you do not need to modify it, but
do read the function signatures, you will be using them throughout the notebook.

What is new since TME2:

- `ml_tree()` builds a **maximum-likelihood** tree with PHYLIP's **`proml`**;
- `align_sequences()` (Clustal Omega, from TME1) now also accepts a list of
  sequences, not only a file;
- `read_species()` and `read_clades()` read `species.list` and `clades.list`;
- `show_tree(tree, colors=...)` colours the leaves by clade, and writes support
  values on the branches; `show_tree(tree, unrooted=True)` draws the tree
  **unrooted**, as NJ and ML build it;
- `check_clades()` tells you, for each clade, whether its species are grouped
  together in a tree;
- `splits()` lists the groupings (splits) of a tree, for the bootstrap.

`read_alignment()`, `distance_matrix()`, `build_tree()`, `sister_group()`,
`compare_topologies()`, `save_newick()` and `embed_image()` are unchanged.
`distance_matrix()` now warns you when two sequences are too different for
`protdist` to estimate a distance.

> **If something looks wrong**, re-run this cell, then the cell that failed. If it
> still fails, read the last line of the error message — it usually names the file
> that could not be found or the program that is missing (`pixi run selftest`
> checks the programs).

In [ ]:
import io, itertools, random, subprocess, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from Bio import SeqIO, AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceMatrix
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from IPython.display import HTML as _HTML, Image as _Image, display

plt.rcParams["figure.dpi"] = 110


# --------------------------------------------------------------------------
# PHYLIP plumbing. You do not need to read this part.
#
# PHYLIP identifies taxa by exactly ten characters, which long names would
# break, so we hand PHYLIP unique tokens and put the real names back
# afterwards. PHYLIP also reads and writes fixed filenames in the current
# directory, so every run gets a private temporary directory.
# --------------------------------------------------------------------------

def _tokens(names):
    return {name: "SEQ%07d" % i for i, name in enumerate(names)}


def _run_phylip(program, infile_text, keystrokes):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "infile").write_text(infile_text)
        proc = subprocess.run([program], input=keystrokes, cwd=tmp,
                              text=True, capture_output=True, timeout=3600)
        outfile, outtree = tmp / "outfile", tmp / "outtree"
        if not outfile.exists():
            raise RuntimeError("%s did not produce an outfile.\n\n%s"
                               % (program, (proc.stdout or proc.stderr)[-2000:]))
        return outfile.read_text(), (outtree.read_text() if outtree.exists() else None)


def _wrap(prefix, values, per_line=6):
    out, chunk = [], [values[i:i + per_line] for i in range(0, len(values), per_line)]
    for i, part in enumerate(chunk):
        head = prefix if i == 0 else " " * len(prefix)
        out.append(head + "".join("%12.6f" % v for v in part))
    return out


def _phylip_alignment(alignment, tok):
    text = [" %d %d" % (len(alignment), alignment.get_alignment_length())]
    text += ["%s%s" % (tok[rec.id], str(rec.seq).upper()) for rec in alignment]
    return "\n".join(text) + "\n"


def _read_newicks(text, back):
    trees = []
    for chunk in text.replace("\n", "").split(";"):
        if chunk.strip():
            tree = Phylo.read(io.StringIO(chunk + ";"), "newick")
            for tip in tree.get_terminals():
                tip.name = back.get((tip.name or "").strip(), tip.name)
            trees.append(tree)
    return trees


def _check_ids(alignment):
    names = [rec.id for rec in alignment]
    if len(set(names)) != len(names):
        dup = sorted({n for n in names if names.count(n) > 1})
        raise ValueError("the same name appears twice in the alignment: %s"
                         % ", ".join(dup[:5]))
    return names


# --------------------------------------------------------------------------
# Sequences and alignments
# --------------------------------------------------------------------------

def read_alignment(path):
    """Read an *already aligned* FASTA file."""
    return AlignIO.read(path, "fasta")


def align_sequences(sequences):
    """Align protein sequences with Clustal Omega.

    sequences : a FASTA file, or a list of SeqRecord (e.g. the output of your
                own selection function). The alignment keeps their `id`s.
    """
    if isinstance(sequences, (str, Path)):
        records = list(SeqIO.parse(sequences, "fasta"))
    else:
        records = [SeqRecord(Seq(str(r.seq)), id=r.id, description="")
                   for r in sequences]
    for rec in records:
        rec.name, rec.description = rec.id, ""
    with tempfile.TemporaryDirectory() as tmp:
        src, dst = Path(tmp) / "in.fasta", Path(tmp) / "out.fasta"
        SeqIO.write(records, src, "fasta")
        subprocess.run(["clustalo", "-i", str(src), "-o", str(dst),
                        "--outfmt=fasta", "--force"],
                       check=True, capture_output=True, text=True, timeout=3600)
        return AlignIO.read(dst, "fasta")


def read_species(path):
    """One name per line -> list of names (blank lines are skipped)."""
    return [l.strip() for l in Path(path).read_text().splitlines() if l.strip()]


def read_clades(path):
    """Read clades.list ('#Clade' header, then one species per line).
    Returns a dict {species: clade}."""
    clade_of, current = {}, None
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line.startswith("#"):
            current = line[1:].strip()
            current = current[:1].upper() + current[1:]
        elif line:
            clade_of[line] = current
    return clade_of


# --------------------------------------------------------------------------
# Distances and trees — PHYLIP `protdist`, `neighbor` and `proml`
# --------------------------------------------------------------------------

def distance_matrix(alignment):
    """Pairwise distances with PHYLIP's `protdist` (JTT model, its default)."""
    names = _check_ids(alignment)
    tok = _tokens(names)
    outfile, _ = _run_phylip("protdist", _phylip_alignment(alignment, tok), "Y\n")

    back = {v: k for k, v in tok.items()}
    fields = outfile.split()
    n, pos = int(fields[0]), 1
    order, rows = [], []
    for _ in range(n):
        order.append(back[fields[pos]]); pos += 1
        rows.append([float(x) for x in fields[pos:pos + n]]); pos += n
    where = {name: i for i, name in enumerate(order)}
    square = [[rows[where[a]][where[b]] for b in names] for a in names]
    undefined = sum(square[i][j] < 0 for i in range(n) for j in range(i))
    if undefined:
        print("note: %d pair(s) of sequences are too different for protdist to "
              "estimate a distance (written -1). Trees built from this matrix "
              "are unreliable." % undefined)
    return DistanceMatrix(names=list(names),
                          matrix=[[square[i][j] for j in range(i + 1)]
                                  for i in range(len(names))])


def build_tree(dm, method="nj"):
    """Build a tree from a distance matrix with PHYLIP's `neighbor`.
    method: 'nj' (Neighbor-Joining) or 'upgma'.
    """
    names = list(dm.names)
    tok = _tokens(names)
    back = {v: k for k, v in tok.items()}
    text = ["   %d" % len(names)]
    for i, name in enumerate(names):
        text += _wrap(tok[name], [dm[i, j] for j in range(len(names))])

    keystrokes = "Y\n" if method == "nj" else "N\nY\n"
    _, outtree = _run_phylip("neighbor", "\n".join(text) + "\n", keystrokes)
    if outtree is None:
        raise RuntimeError("neighbor produced no tree")
    tree = _read_newicks(outtree, back)[0]
    if method == "nj":
        tree.root_at_midpoint()     # NJ is unrooted; midpoint is only for drawing
    return tree


def ml_tree(alignment):
    """Maximum-likelihood tree with PHYLIP's `proml` (JTT model, its defaults).

    Returns the tree; its log-likelihood is in `tree.log_likelihood`.
    `proml` is much slower than NJ: seconds for one domain, about a minute for
    a few thousand columns.
    """
    names = _check_ids(alignment)
    tok = _tokens(names)
    back = {v: k for k, v in tok.items()}
    outfile, outtree = _run_phylip("proml", _phylip_alignment(alignment, tok), "Y\n")
    if outtree is None:
        raise RuntimeError("proml produced no tree")
    tree = _read_newicks(outtree, back)[0]
    tree.log_likelihood = None
    for line in outfile.splitlines():
        if "Ln Likelihood" in line:
            tree.log_likelihood = float(line.split("=")[1])
    tree.root_at_midpoint()         # proml's tree is unrooted; midpoint is for drawing
    return tree


# --------------------------------------------------------------------------
# Reading and comparing trees
# --------------------------------------------------------------------------

def negative_branches(tree):
    """List the negative branch lengths in a tree, most negative first."""
    out = [(c.name or "(internal)", c.branch_length)
           for c in tree.find_clades()
           if c.branch_length is not None and c.branch_length < 0]
    return sorted(out, key=lambda x: x[1])


def sister_group(tree, taxa):
    """Smallest clade of `tree` containing at least two of `taxa`.

    Returns (which of `taxa` are in it, all its leaves).
    """
    taxa = set(taxa)
    best = None
    for clade in tree.find_clades():
        leaves = {t.name for t in clade.get_terminals()}
        if len(leaves & taxa) >= 2 and (best is None or len(leaves) < len(best)):
            best = leaves
    if best is None:
        return None, None
    return sorted(best & taxa), sorted(best)


def _splits(tree, leaves, min_length):
    """Non-trivial bipartitions of an (unrooted) tree, restricted to `leaves`,
    each mapped to the clades of `tree` that carry it."""
    leaves = frozenset(leaves)
    anchor = min(leaves)
    out = {}
    for clade in tree.find_clades():
        if clade is tree.root or (clade.branch_length is not None
                                  and clade.branch_length <= min_length):
            continue
        side = frozenset(t.name for t in clade.get_terminals()) & leaves
        if anchor in side:
            side = leaves - side
        if 1 < len(side) < len(leaves) - 1:
            out.setdefault(side, []).append(clade)
    return out


def splits(tree):
    """The splits (bipartitions) of a tree, ignoring where it is rooted.

    Cutting an internal branch divides the leaves in two groups: that is a
    split. Each split is given as one of its two groups (a frozenset of leaf
    names, always the side *without* the alphabetically first leaf), so the
    same split is written the same way in every tree with the same leaves.

    Returns a dict {split: [clades of `tree` on that branch]} — usually one
    clade; two when the split is the branch the drawing root was placed on.
    """
    return _splits(tree, {t.name for t in tree.get_terminals()}, min_length=-np.inf)


def compare_topologies(tree1, tree2, min_length=1e-3):
    """Robinson-Foulds comparison of two trees on the leaves they share.

    Only branches longer than `min_length` count — near-zero branches join
    identical or almost identical sequences in an arbitrary order and support
    nothing. Returns a dict: number of shared leaves, number of supported splits
    in each tree, how many are found in both, and the RF distance (splits found
    in only one of the two trees). RF = 0 means the same unrooted topology.
    """
    common = ({t.name for t in tree1.get_terminals()}
              & {t.name for t in tree2.get_terminals()})
    s1 = set(_splits(tree1, common, min_length))
    s2 = set(_splits(tree2, common, min_length))
    return {"shared leaves": len(common), "splits tree 1": len(s1),
            "splits tree 2": len(s2), "shared splits": len(s1 & s2),
            "RF distance": len(s1 ^ s2)}


def check_clades(tree, clade_of):
    """For each clade of `clade_of` ({species: clade}), is it grouped together
    in `tree`? The tree is read as unrooted: a clade is grouped together when
    one branch separates exactly its species from all the others.

    Returns a table: number of species, grouped together or not, and the
    smallest group of the tree that contains all of them, with the species
    from other clades that it also contains ("intruders").
    """
    leaves = frozenset(t.name for t in tree.get_terminals())
    groups = {leaves}
    for clade in tree.find_clades():
        side = frozenset(t.name for t in clade.get_terminals())
        groups |= {side, leaves - side}
    rows = []
    for name in sorted(set(clade_of.values())):
        members = frozenset(s for s, c in clade_of.items() if c == name) & leaves
        if not members:
            continue
        if len(members) == 1:
            rows.append({"clade": name, "species": 1, "grouped together": "—",
                         "smallest group": 1, "intruders": ""})
            continue
        best = min((g for g in groups if members <= g), key=len)
        rows.append({"clade": name, "species": len(members),
                     "grouped together": "yes" if best == members else "no",
                     "smallest group": len(best),
                     "intruders": ", ".join(sorted(best - members))})
    return pd.DataFrame(rows).set_index("clade")


def save_newick(tree, path):
    """Write the tree in Newick format (e.g. to upload it to iTOL).
    Support values stored in `clade.confidence` are written too."""
    Phylo.write(tree, path, "newick")
    print("wrote", path)


# --------------------------------------------------------------------------
# Display
# --------------------------------------------------------------------------

def as_dataframe(dm, decimals=3):
    """Distance matrix as a pandas table, so it displays nicely in the notebook."""
    n = len(dm.names)
    full = [[dm[i, j] for j in range(n)] for i in range(n)]
    return pd.DataFrame(full, index=dm.names, columns=dm.names).round(decimals)


_PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e",
            "#17becf", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22"]


def _segment_gap(p1, p2, q1, q2):
    """Smallest distance between segments p1-p2 and q1-q2 (2-D numpy points)."""
    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
    if (cross(p1, p2, q1) * cross(p1, p2, q2) < 0
            and cross(q1, q2, p1) * cross(q1, q2, p2) < 0):
        return 0.0                               # they cross
    def to_segment(p, a, b):
        ab = b - a
        t = np.clip(np.dot(p - a, ab) / max(np.dot(ab, ab), 1e-12), 0, 1)
        return np.linalg.norm(p - (a + t * ab))
    return min(to_segment(p1, q1, q2), to_segment(p2, q1, q2),
               to_segment(q1, p1, p2), to_segment(q2, p1, p2))


def _draw_unrooted(tree, ax, label_color, branch_color, fontsize=8):
    """Equal-angle layout: each subtree gets a wedge of the circle proportional
    to its number of leaves, and each branch points to the middle of its wedge.
    The root of the drawing is not shown: at a two-child root, the two branches
    leave in opposite directions and read as one straight branch.

    Labels are not allowed to overlap: a label that would touch one already
    placed slides outward along its own direction, joined to its leaf by a
    dotted line. Branch lengths are never altered."""
    from matplotlib.textpath import TextPath

    size = {}
    for clade in tree.find_clades(order="postorder"):
        size[clade] = 1 if clade.is_terminal() else sum(size[c] for c in clade.clades)
    points, leaves = [], []

    def place(clade, x, y, a0, a1):
        start = a0
        for child in clade.clades:
            width = (a1 - a0) * size[child] / size[clade]
            mid = start + width / 2
            length = max(child.branch_length or 0.0, 0.0)
            cx, cy = x + length * np.cos(mid), y + length * np.sin(mid)
            ax.plot([x, cx], [y, cy], color=branch_color(child), lw=1.2,
                    solid_capstyle="round")
            points.append((cx, cy))
            if child.is_terminal():
                leaves.append((child.name, cx, cy, mid))
            elif child.confidence is not None:
                ax.text(cx, cy, " %g" % child.confidence, fontsize=fontsize - 1.5,
                        color="dimgray", va="bottom", ha="left")
            place(child, cx, cy, start, start + width)
            start += width

    place(tree.root, 0.0, 0.0, 0.0, 2 * np.pi)
    xs, ys = zip(*points)
    span = max(max(xs) - min(xs), max(ys) - min(ys)) or 1.0
    pad = 0.28 * span                            # room for the labels
    ax.set_xlim(min(xs) - pad, max(xs) + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + pad)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.apply_aspect()                            # fixes the data -> pixel scale

    # Place the labels, in pixels, so that no two of them overlap.
    to_px = ax.transData
    px_per_unit = to_px.transform((1, 0))[0] - to_px.transform((0, 0))[0]
    pt = ax.figure.dpi / 72
    height = 1.15 * fontsize * pt                # thickness of a label
    base, step = 3 * pt, 0.5 * height
    placed = []
    for name, x, y, angle in sorted(leaves, key=lambda leaf: leaf[3]):
        u = np.array([np.cos(angle), np.sin(angle)])
        origin = np.array(to_px.transform((x, y)))
        length = TextPath((0, 0), str(name), size=fontsize).get_extents().width * pt
        offset = base
        for _ in range(400):
            a, b = origin + offset * u, origin + (offset + length) * u
            if all(_segment_gap(a, b, c, d) >= height for c, d in placed):
                break
            offset += step
        placed.append((a, b))
        ax_, ay_ = x + offset / px_per_unit * u[0], y + offset / px_per_unit * u[1]
        color = label_color(name)
        if offset > base:
            ax.plot([x, ax_], [y, ay_], color=color, lw=0.6, ls=":", alpha=0.7)
        deg = np.degrees(angle) % 360
        flip = 90 < deg < 270
        ax.text(ax_, ay_, str(name), fontsize=fontsize, color=color, va="center",
                ha="right" if flip else "left",
                rotation=deg + 180 if flip else deg, rotation_mode="anchor")

    bar = float("%.0e" % (span / 8))             # scale bar, one significant digit
    x0, y0 = min(xs) - pad * 0.6, min(ys) - pad * 0.6
    ax.plot([x0, x0 + bar], [y0, y0], color="black", lw=1.5)
    ax.text(x0 + bar / 2, y0, "%g\n" % bar, ha="center", va="bottom", fontsize=fontsize)


def show_tree(tree, title="", highlight=(), colors=None, figsize=None, unrooted=False):
    """Draw a tree inline.

    highlight : names coloured red.
    colors    : a dict {leaf name: group}, e.g. the output of read_clades();
                each group gets its own colour, with a legend.
    unrooted  : False draws the tree rooted, as PHYLIP returns it or at the
                midpoint (NJ, proml); True draws it **unrooted**, as a star of
                branches — the honest picture of an NJ or ML tree. In that
                drawing a branch takes the colour of a group when every leaf
                beyond it belongs to that group, and the names of (nearly)
                identical sequences are pushed apart, with a dotted line to
                their leaf.
    Support values stored in `clade.confidence` are written on the branches.
    """
    n = len(tree.get_terminals())
    default = (9, 9) if unrooted else (9, max(2.5, 0.22 * n + 1))
    fig, ax = plt.subplots(figsize=figsize or default)
    hl = set(highlight)
    groups = sorted(set(colors.values())) if colors else []
    shade = {g: _PALETTE[i % len(_PALETTE)] for i, g in enumerate(groups)}

    def label_color(name):
        if name in hl:
            return "crimson"
        if colors and name in colors:
            return shade[colors[name]]
        return "black"

    def branch_color(clade):
        found = {colors.get(t.name) for t in clade.get_terminals()} if colors else {None}
        return shade[found.pop()] if len(found) == 1 and None not in found else "0.35"

    if unrooted:
        _draw_unrooted(tree, ax, label_color, branch_color)
    else:
        Phylo.draw(tree, axes=ax, do_show=False, label_colors=label_color)
        ax.set_ylabel("")
        ax.set_yticks([])
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)

    # Title above, legend to the right of everything drawn — never on the tree.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_window_extent(renderer)] + [t.get_window_extent(renderer) for t in ax.texts]
    top = max(b.y1 for b in boxes)
    right = max(b.x1 for b in boxes)
    ax.set_title(title, y=max(1.0, ax.transAxes.inverted().transform((0, top))[1] + 0.01))
    if groups:
        anchor = fig.transFigure.inverted().transform((right + 10, ax.get_window_extent(renderer).y1))
        fig.legend(handles=[Patch(color=shade[g], label=g) for g in groups],
                   loc="upper left", bbox_to_anchor=tuple(anchor),
                   fontsize=8, frameon=False)
    plt.show()
    neg = negative_branches(tree)
    if neg:
        print("note: %d branch(es) have a negative length (most negative: %.4f). "
              "This is normal for NJ." % (len(neg), neg[0][1]))


def embed_image(path, max_width=1000, caption=""):
    """Put an image file *inside* the notebook, so it travels with the .ipynb.

    Use this for screenshots or figures downloaded from a website (PNG, JPEG
    or SVG — e.g. a tree drawn in iTOL). Once the cell has run, the original
    file is no longer needed by anyone reading your notebook.
    """
    if str(path).lower().endswith(".svg"):
        data = Path(path).read_text()
        display(_HTML('<div style="max-width:100%%;overflow-x:auto">%s</div>' % data))
        print("%s  (SVG, %.0f KB embedded in this notebook)"
              % (caption or path, len(data) / 1024))
        return
    from PIL import Image as _PIL
    im = _PIL.open(path)
    if im.width > max_width:
        im = im.resize((max_width, round(im.height * max_width / im.width)),
                       _PIL.LANCZOS)
    buf = io.BytesIO()
    if im.mode in ("RGBA", "LA", "P"):
        im.convert("RGBA").save(buf, format="PNG", optimize=True)
    else:
        im.convert("RGB").save(buf, format="JPEG", quality=85, optimize=True)
    display(_Image(data=buf.getvalue()))
    print("%s  (%dx%d, %.0f KB embedded in this notebook)"
          % (caption or path, im.width, im.height, len(buf.getvalue()) / 1024))


print("Toolbox loaded.")

***
## From the lecture

Definitions the questions below assume. None of this is an answer.

**Pfam domain family.** A *domain* is a part of a protein that folds and evolves
as a unit. [Pfam](https://www.ebi.ac.uk/interpro/) groups the domains that are
**homologous** — descended from a common ancestral domain — into families, each
with an identifier such as `PF01599` (*Ribosomal_S27*). Each file in
`files/TME4_sequences/` holds the domain sequences of one family, named
`ACCESSION_SPECIES`: `Q6C290_YARLI` is UniProt entry Q6C290, from *Yarrowia
lipolytica*. The species codes are UniProt's.

**Maximum likelihood (ML).** Given an alignment and a model of amino-acid
substitution (`proml` uses JTT, as `protdist` does), the *likelihood* of a tree is
the probability of the alignment if the sequences had evolved along that tree.
ML looks for the tree, with its branch lengths, of highest likelihood. Unlike NJ,
which only sees the matrix of pairwise distances, ML uses every column of the
alignment directly. Searching tree space is heuristic (TME2, Q1.5) and slow.

**Split.** Cutting one internal branch of a tree divides its leaves in two groups:
a *split* (or bipartition). An unrooted tree is entirely described by its set of
splits. A set of species is **grouped together** in an unrooted tree when one
split separates exactly those species from all the others — wherever the root is
eventually placed, they then form a clade, or the complement of one.

**Bootstrap** (Felsenstein, 1985). The alignment columns are treated as a sample
of the sites of the genome. A *bootstrap replicate* is a new alignment with the
same number of columns, drawn **at random with replacement** from the original
columns — some columns appear twice or more, others not at all. A tree is built
from each replicate in the same way as the original tree. The **bootstrap
support** of a branch of the original tree is the percentage of replicate trees
that contain its split. It is computed on unrooted trees.

*To go further:* the lectures *Phylogenetic reconstruction based on multiple
genes* and *Bootstrap methods in phylogeny* (A. Carbone).

***
## The data

Exercises 2 to 4 use the same species. `files/species.list`
names **40 eukaryotes**; `files/clades.list` sorts them into **7 clades**
(it also names a few species that are not in `species.list` — ignore them).

In [ ]:
species = read_species("files/species.list")
clade_of = read_clades("files/clades.list")
print(len(species), "species")
pd.Series([clade_of[s] for s in species]).value_counts().rename("species").to_frame()

***
# Exercise 1 — Large-scale phylogenies

**Q1.1** What are the problems related to the construction of phylogenetic trees
with **thousands of species**? What are the strategies for reconstructing
phylogenetic trees on a large scale? How is it possible to handle **data
fragmentation**?

### &#9997;&#65039; Your answer — Q1.1

_Double-click this cell and replace this line with your answer._

***
# Exercise 2 — Phylogenetic tree from a single domain family

**Q2.1** Get the *Ribosomal_S27* (**PF01599**) sequences from
`files/TME4_sequences/PF01599.fasta` and select only those belonging to the
species listed in `species.list`. If there are several sequences for a species,
just take the first one. Then align the sequences and build two trees: one with
`protdist` + **`neighbor`** (NJ, as in TME1) and one with **`proml`** (maximum
likelihood).

Name each selected sequence by its **species code** only (`YARLI`, not
`Q6C290_YARLI`): the trees must use the names of `species.list` and
`clades.list`, and in Exercise 4 the same species must have the same name in every
family.

In [ ]:
# ---- TODO 1/2 ----
# Complete one_sequence_per_species(path, species): it returns a list of
# SeqRecord, one per species of `species`, each renamed with its species code
# (id = "YARLI"). You will reuse it for every family in Exercise 4.
# The first and last lines are given: only write the body of the for loop.
#
# SeqIO.parse(path, "fasta") iterates over the records; rec.id is the name
# ("Q6C290_YARLI"); SeqRecord(rec.seq, id="YARLI", description="") makes a renamed copy.

# Uncomment the function template below (select its lines, then Ctrl+/)
# and start from here.

# def one_sequence_per_species(path, species):
#     found = {}                                   # species code -> renamed SeqRecord
#     for rec in SeqIO.parse(path, "fasta"):
#         # ---- write the body of this loop ----
#         # 1. get the species code from rec.id ("Q6C290_YARLI" -> "YARLI")
#         # 2. if that code is in `species` (and not already in `found`: keep the
#         #    first one), store a renamed copy of rec in found[code]
#         ...
#     return [found[s] for s in species if s in found]

In [ ]:
# ---- TODO 2/2 ----
# Select the PF01599 sequences, align them with align_sequences(), and build
#   pf_nj = the NJ tree        (distance_matrix + build_tree)
#   pf_ml = the ML tree        (ml_tree — a few seconds)
# Expected: 40 sequences, about 50 alignment columns.

**Q2.2** Show the two trees, with **a different colour for each clade**
(`clades.list`), and compare them. Were the clades grouped together? Which ones,
in which tree?

> `show_tree(tree, colors=clade_of)` colours the leaves. `check_clades(tree, clade_of)`
> tells you, clade by clade, whether its species are grouped together, and which
> other species get in the way. `compare_topologies(tree1, tree2)` (TME2) counts
> the splits two trees share.
>
> Both trees are **unrooted**. By default `show_tree()` draws them rooted at the
> midpoint, which is only a way of drawing them: a clade may then look split in
> two on the figure and still be grouped together. `show_tree(tree, colors=clade_of,
> unrooted=True)` draws the tree as it really is, without a root — a clade is
> grouped together when it occupies a sector of the drawing that one branch
> cuts off. `check_clades()` gives the same answer without looking.

In [ ]:
# ---- TODO ----
# Show pf_nj and pf_ml coloured by clade, run check_clades() on each, and
# compare_topologies(pf_nj, pf_ml).

### &#9997;&#65039; Your answer — Q2.2

_Double-click this cell and replace this line with your answer._

***
# Exercise 3 — Bootstrap

The goal of this exercise is to compute the bootstrap support of each branch of
the **`proml`** tree of Exercise 2 (`pf_ml`).

**Q3.1** Build **10** bootstrap alignments by sampling columns of the PF01599
alignment **with replacement**, until each bootstrap alignment has as many columns
as the original one.

> **10 replicates is very few**: published analyses use 100 to 1000. With 10,
> every support value is a multiple of 10 %, so read them as rough indications.
> We keep 10 here only because each replicate costs one `proml` run.

In [ ]:
# ---- TODO ----
# Complete bootstrap_one_replicate(alignment, rng): it returns ONE bootstrap
# replicate, a MultipleSeqAlignment with the same sequences (same ids) as
# `alignment` and n columns drawn at random with replacement. The first and
# last lines are given.
# Then build `replicates`, a list of 10 replicates of pf_aln.
#
# rng.randrange(n) draws a random integer in 0 .. n-1;
# alignment[:, i:i + 1] is column i, as a one-column alignment, and two
# alignments of the same sequences can be joined side by side with `+`.
# (Or build each sequence as a string and make a new MultipleSeqAlignment.)

# Uncomment the function template below (select its lines, then Ctrl+/)
# and start from here.

# def bootstrap_one_replicate(alignment, rng):
#     n = alignment.get_alignment_length()         # number of columns
#     # ---- write here the code that builds `replicate` ----
#     ...
#     return replicate

# # A fixed seed makes your replicates reproducible:
# rng = random.Random(2026)

**Q3.2** For each bootstrap alignment, build a tree with `proml` (`ml_tree()`).

In [ ]:
# ---- TODO ----
# boot_trees = a list of 10 ML trees, one per replicate (about 15 s in all).

**Q3.3** Compute the bootstrap support of each branch of the original tree
`pf_ml`, annotate the tree with it, show it and save it in Newick format.

> `splits(tree)` returns the splits of a tree as a dict `{split: [clades]}`, the
> split written the same way in every tree. A split of `pf_ml` is supported by a
> bootstrap tree when it is also a split of that tree. Store the support (in %)
> in `clade.confidence`: `show_tree()` writes it on the branch, `save_newick()`
> writes it in the file.

In [ ]:
# ---- TODO ----
# reference = {split: [clades]}: its clades ARE branches of pf_ml (not copies),
# so setting clade.confidence writes the support onto pf_ml itself; show_tree()
# and save_newick() then display / save it — nothing else to pass to them.
# The loops and the last lines are given: only write the code that computes
# `support`.

# Uncomment the template below (select its lines, then Ctrl+/) and start
# from here.

# reference = splits(pf_ml)
# for split, clades in reference.items():
#     # ---- write here the code that computes `support` ----
#     # count the trees of boot_trees that also have this split
#     # (`split in splits(t)` is True when tree t has it), then turn the count
#     # into a percentage of len(boot_trees).
#     ...
#     for clade in clades:        # usually one clade, two for the root branch
#         clade.confidence = support

# show_tree(pf_ml, "PF01599 — proml tree with bootstrap support", colors=clade_of)
# save_newick(pf_ml, "PF01599_proml_bootstrap.nwk")

**Q3.4** Which branches are well supported? Are the clades you found grouped
together in Q2.2 among them? What does a bootstrap value measure — and what does it
*not* tell you? Is 10 replicates enough?

### &#9997;&#65039; Your answer — Q3.4

_Double-click this cell and replace this line with your answer._

***
# Exercise 4 — Phylogenetic tree from multiple domain families

**Q4.1** Consider **all** the Pfam families of `files/TME4_sequences/`. For each
family, select the sequences of the species of `species.list`, as in Exercise 2.
If a family does **not** contain **all** 40 species, do not consider it. How many
families are left?

In [ ]:
# ---- TODO ----
# Reuse one_sequence_per_species() on every family file, and keep in
#     families = {"PF01599": [records], ...}      (Path(...).stem is "PF01599")
# only the families that have all 40 species.
# A few files are empty: your loop must not fail on them.
# Then print how many families are left (Q4.1).

# Uncomment the template below (select its lines, then Ctrl+/) and start
# from here.

# families = {}
# for path in sorted(Path("files/TME4_sequences").glob("*.fasta")):
#     # ---- write the body of this loop ----
#     ...

**Q4.2** Align the sequences of each selected family, **concatenate** the
alignments into one, and build the NJ and the `proml` tree of the concatenated
alignment. Show them, coloured by clade. Were the clades grouped together? Compare
with Exercise 2.

> To concatenate, the sequences must be in the **same order** in every alignment:
> `aln.sort()` sorts an alignment by sequence name, in place. `aln1 + aln2` then
> joins two alignments side by side. Keep the separate alignments too, in a dict
> `family_alns = {"PF01599": alignment, ...}` — you need them in the bonus (Q4.3).
>
> `proml` on the concatenated alignment takes **about a minute** (a few minutes
> on an older laptop).

In [ ]:
# ---- TODO ----
# family_alns = {family: its alignment (sorted)}
# concat      = all of them joined side by side
# cat_nj, cat_ml = the NJ and ML trees of concat; show them; check_clades() on each.
# Expected: 40 sequences, about 2 800 columns.

### &#9997;&#65039; Your answer — Q4.2

_Double-click this cell and replace this line with your answer._

***
## Bonus (optional) — why is the concatenated tree not better?

More data should give a better tree, not a worse one: something is wrong with
some of these families. Q4.3 and Q4.4 find what, and fix it. They are
**optional**: if you are short of time, skip them and go to Exercise 5.

Concatenating assumes that every family follows the **same history** — the
history of the species: in every family, closely related species have nearly
identical sequences, and distant species different ones. A family where this is
not true does not follow the species tree, and pulls the concatenated tree away
from it.

To check this, look at species that are **very closely related** — they diverged
a few tens of millions of years ago at most, so their ribosomal proteins should
be almost identical, in every family:

In [ ]:
close_relatives = {
    "mammals":        ["HUMAN", "MACFA", "MOUSE", "RAT"],
    "Drosophila":     ["DROME", "DROSE", "DROER", "DROMO"],
    "Plasmodium":     ["PLAF7", "PLABE", "PLAVI", "PLAKH", "PLAYO", "PLACH"],
    "Caenorhabditis": ["CAEEL", "CAEBR"],
}

**Q4.3 (bonus)** Start with one family, PF00297. The cell below shows the
distances between the four mammals (it is given: uncomment it and run it):

In [ ]:
# Bonus (given code): uncomment the code below (select its lines, then
# Ctrl+/) and run it.

# dm = distance_matrix(family_alns["PF00297"])
# mammals = close_relatives["mammals"]
# pd.DataFrame([[dm[a, b] for b in mammals] for a in mammals],
#              index=mammals, columns=mammals).round(2)

Human and macaque are almost identical, and so are mouse and rat — but the two
pairs are as far apart as unrelated sequences of the family. The cell below
(given too, uncomment it) does the same check for every family and every group:
for each group, it shows the **largest distance** between two of its species.

> `distance_matrix()` warns, for a few families, that some pairs are too
> different for `protdist` to estimate a distance (written -1). You can ignore
> it here: the only close relatives concerned (*Plasmodium* in PF01201) are in
> a family that fails the test anyway.

In [ ]:
# Bonus (given code): uncomment the code below (select its lines, then
# Ctrl+/) and run it.

# def largest_distance(dm, names):
#     """The largest distance between two of `names` in the distance matrix `dm`."""
#     return max(dm[a, b] for a, b in itertools.combinations(names, 2))

# rows = {}
# for name, aln in family_alns.items():
#     dm = distance_matrix(aln)
#     rows[name] = {group: largest_distance(dm, members)
#                   for group, members in close_relatives.items()}
# table = pd.DataFrame(rows).T.round(2)       # one row per family
# table

A family **passes the test** when every group is close: largest distance
**below 0.2**. Which families fail? What can explain that close relatives are so
far apart in these families?

> Hint: in a eukaryotic cell, ribosomes are found in more than one place.

### &#9997;&#65039; Your answer — Q4.3

_Double-click this cell and replace this line with your answer._

**Q4.4 (bonus)** Keep only the families that pass the test, concatenate their
alignments, and build the NJ and `proml` trees again. Show them, coloured by
clade. Were the clades grouped together now? Compare with Q4.2 and with
Exercise 2.

In [ ]:
# ---- TODO (bonus) ----
# clean = the names of the families that pass the test (every group below 0.2).
# table.loc[name] is the row of family `name`, and table.loc[name].max() its
# largest value.
# Then concatenate the alignments of `clean` (family_alns) as in Q4.2, and
# build, show and check_clades() the NJ and ML trees: clean_nj, clean_ml.

# Uncomment the template below (select its lines, then Ctrl+/) and start
# from here.

# clean = []
# for name in table.index:
#     # ---- add `name` to clean if the family passes the test ----
#     ...

### &#9997;&#65039; Your answer — Q4.4

_Double-click this cell and replace this line with your answer._

***
# Exercise 5 — Coronaviridae: one tree per gene

In Exercise 4 you combined many genes into one tree. Here you do the opposite:
build one tree **per gene** for the same viruses, and ask whether they agree.
Following Exercise 5 of TME1, read the references at the end of the notebook and
answer the questions below.

**Q5.1** What is the role of the **nucleocapsid (N)** protein?

**Q5.2** What is the role of the **membrane (M)** protein?

**Q5.3** What is the role of the **envelope (E)** protein?

### &#9997;&#65039; Your answer — Q5.1 to Q5.3

_Double-click this cell and replace this line with your answer._

**Q5.4** `files/N_protein.afa`, `files/M_protein.afa` and `files/E_protein.afa`
are alignments of the N, M and E proteins across the same set of coronaviruses.
Reconstruct (`protdist` + Neighbor-Joining, as in TME1) and show the three trees.

In [ ]:
# ---- TODO ----
# For each protein: read the alignment (read_alignment), compute distances,
# build the NJ tree and show it. Keep the three trees in `cov_trees`:
# you will compare them in Q5.5.
# Expected: N 50 sequences x 422, M 50 x 227, E 49 x 76.

# Uncomment the template below (select its lines, then Ctrl+/) and start
# from here.

# cov_trees = {}                                # "N" -> the NJ tree of N, ...
# for prot in ("N", "M", "E"):
#     aln = read_alignment("files/%s_protein.afa" % prot)
#     # ---- write the body of this loop ----
#     ...

**Q5.5** Is there a **consensus** between these trees?

> `compare_topologies(tree1, tree2)` counts the groupings (splits) two trees share.
> Branches of (almost) zero length join identical sequences in an arbitrary order,
> so they are ignored. Use `sister_group()` and `highlight=` to look at specific
> viruses — for instance SARS-CoV-2 and SARS-CoV-1 (`Human_SARS_...`).
>
> "Consensus" has a precise meaning in phylogenetics: a **consensus tree**
> (strict, or majority-rule) keeps only the groupings found in all, or most, of
> the trees. Here we stay simpler and compare the trees two by two with the
> Robinson–Foulds count of `compare_topologies()`. Other measures exist, for
> example the **MAST** (maximum agreement subtree): the largest set of leaves on
> which two trees agree exactly, which tells how many taxa you would have to
> remove for the trees to agree.

In [ ]:
# ---- TODO ----
# Compare the three trees two by two, and look at where SARS-CoV-2 and SARS-CoV-1
# fall in each.

### &#9997;&#65039; Your answer — Q5.5

_Double-click this cell and replace this line with your answer._

**Q5.6** In Exercise 4, you combined genes by concatenation. Would it make sense
to concatenate the N, M and E alignments into one? What would you need to check
first?

> Hint: the problem here is not the one of Exercise 4 (see its bonus): N, M and
> E are the same three genes in every virus. What else must be true of the genes
> for a concatenation to make sense? Think of your answer to Q5.5.

### &#9997;&#65039; Your answer — Q5.6

_Double-click this cell and replace this line with your answer._

***
## References

1. Cubuk, J. *et al.* "The SARS-CoV-2 nucleocapsid protein is dynamic, disordered,
   and phase separates with RNA." *Nature Communications* **12**, 1936 (2021).
2. Thomas, S. "The structure of the membrane protein of SARS-CoV-2 resembles the
   sugar transporter SemiSWEET." *Pathogens and Immunity* **5**(1), 342–363 (2020).
3. Chai, J. *et al.* "Structural basis for SARS-CoV-2 envelope protein recognition
   of human cell junction protein PALS1." *Nature Communications* **12**, 3433 (2021).
4. Felsenstein, J. "Confidence limits on phylogenies: an approach using the
   bootstrap." *Evolution* **39**(4), 783–791 (1985).